# Synthetic Organizational Document Generation for Brazilian ESG Companies

This notebook implements a synthetic data generation pipeline combining concepts from:

- Survey Paper: "On LLMs-Driven Synthetic Data Generation, Curation, and Evaluation" (Lin Long et al.)
- DocGenie Paper: "DocGenie: A Framework for High-Fidelity Synthetic Document Generation" (Harikrishnan P M et al.)

## 1. Imports and Setup

In [81]:
import os
import json
import requests
import random
from typing import List, Dict, Any
from datetime import datetime
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

from dotenv import load_dotenv
load_dotenv()

# Set random seeds for reproducibility
random.seed(42)

/home/nate/Documents/dev/dss-se-multi-agent-system/synthetic-data-generation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Model loading
Using the GAIA model (Gemma-3-Gaia-PT-BR-4b), a Brazilian Portuguese Multimodal LLM, via Ollama, which runs on a remote server.

In [31]:
class OllamaClient:
    """
    Client wrapper for Ollama server.
    """

    def __init__(self, server_url=None, model_name="brunoconterato/Gemma-3-Gaia-PT-BR-4b-it:f16"):
        """
        Initialize Ollama client.
        
        Args:
            server_url: Desktop IP (e.g., "http://192.168.1.100:11434")
                       If None, uses OLLAMA_SERVER env variable or localhost
            model_name: Ollama model to use
        """
        # Get server URL from env or parameter
        self.server_url = (
            server_url 
            or os.getenv("OLLAMA_SERVER", "http://localhost:11434")
        )
        if not self.server_url.startswith("http"):
            self.server_url = f"http://{self.server_url}"


        self.model_name = model_name
        self.api_url = f"{self.server_url}/api/generate"

    def generate(self, prompt, max_tokens=1024, temperature=0.7, stream=False):
        """
        Generate text using Ollama server.
        
        Args:
            prompt: Input text prompt
            max_tokens: Maximum tokens to generate
            temperature: Sampling temperature (0.0-1.0)
            stream: Whether to stream response
        
        Returns:
            Generated text string
        """
        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": stream,
            "options": {
                "num_predict": max_tokens,
                "temperature": temperature,
                "top_p": 0.9
            }
        }
        
        try:
            response = requests.post(
                self.api_url, 
                json=payload,
                timeout=120  # 2 minute timeout
            )
            
            if response.status_code == 200:
                return response.json()["response"]
            else:
                raise Exception(
                    f"Generation failed: {response.status_code} - {response.text}"
                )

        except requests.exceptions.Timeout:
            print("Generation timed out. Try reducing max_tokens.")
            raise
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            raise

def load_gaia_model():
    """
    Initialize the GAIA model client via Ollama.
    
    Returns:
        OllamaClient: Client ready for inference
    """
    print("Initializing GAIA model client...")
    
    # The client will use OLLAMA_SERVER env variable if set
    # Otherwise defaults to localhost
    client = OllamaClient()
    
    # Test connection
    try:
        test_response = client.generate("Test", max_tokens=10, temperature=0.1)
        print(f"✓ Connection successful! Server: {client.server_url}")
        print(f"✓ Model: {client.model_name}")
        return client
    except Exception as e:
        print(f"✗ Connection failed: {e}")
        print(f"  Make sure Ollama is running at {client.server_url}")
        print(f"  Set OLLAMA_SERVER environment variable if using remote server")
        raise

# Load the model
model_client = load_gaia_model()

Initializing GAIA model client...
✓ Connection successful! Server: http://192.168.18.9:11434
✓ Model: brunoconterato/Gemma-3-Gaia-PT-BR-4b-it:f16


## 3. Load Seed Data
The data was manually taken from Sistema B database. Link: https://www.bcorporation.net/en-us/find-a-b-corp/?refinement%5BhqCountry%5D%5B0%5D=Brazil

In [ ]:
file_path = 'seed_data/companies_mvv.json'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        # Load the JSON data from the file into a Python dictionary
        seed_companies = json.load(f)
    
    print("Successfully loaded data")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

print(seed_companies)

Successfully loaded data
[{'company': 'Ecovalor Consultoria e Assessoria em Sustentabiliade LTDA', 'sector': 'ESG Consulting', 'mission': 'Guiar organizações rumo às melhores práticas de sustentabilidade, atuando em todo Brasil e oferecendo suporte a empresas de diversos setores e tamanhos.', 'vision': None, 'values': ['Conectar', 'Solucionar', 'Inovar', 'Re(voluir)']}, {'company': 'Florestal Alto Uruguai', 'sector': 'ESG Consulting', 'mission': 'Entregar sustentabilidade com acurácia, inovação e simplicidade.', 'vision': 'Ser referência nos setores em que atuamos, admirada por nossos clientes e lembrada por fazer diferente.', 'values': ['Confiança', 'Transparência', 'Encantamento']}, {'company': 'Weber Ambiental', 'sector': 'ESG Consulting', 'mission': 'Contribuir para a melhoria da condição ambiental e social do país, incluindo áreas de vulnerabilidade, proporcionando saúde e qualidade de vida.', 'vision': 'Até 2030, ser reconhecida nacionalmente pela atuação e, principalmente, inova

## 4. Generate Unified Company Profile

To ensure cross-document coherence (Survey paper: sample-wise decomposition strategy)

In [55]:
def create_company_profile_prompt(seed_companies: List[Dict]) -> str:
    """
    Create prompt to generate a unified company profile.
    
    Args:
        seed_companies: Seed company data
    
    Returns:
        Formatted prompt string
    """
    
    task_spec = """Você é um especialista em criar perfis organizacionais para empresas 
brasileiras de consultoria ESG. Sua tarefa é gerar um perfil sintético coerente e realista 
que será usado como base para criar documentos estratégicos."""
    
    # Seed context
    seed_section = "\n\n=== EMPRESAS DE REFERÊNCIA (Setor ESG no Brasil) ===\n"
    
    for i, company in enumerate(seed_companies, 1):
        seed_section += f"\n{i}. {company['company']}\n"
        seed_section += f"   Missão: {company['mission'][:100]}...\n"
        if company['vision']:
            seed_section += f"   Visão: {company['vision'][:100]}...\n"
    
    # Instructions
    instructions = """

=== TAREFA ===

Gere um perfil completo para UMA empresa sintética de consultoria ESG brasileira.
Este perfil deve ser internamente consistente e realista.

GERE O SEGUINTE PERFIL:

**Nome da Empresa:** [Nome criativo e realista, formato: "Nome + Consultoria/Soluções"]

**Localização:** [Cidade e Estado brasileiro, inspirado nas empresas semente]

**Ano de Fundação:** [Entre 2015-2020, típico do setor]

**Porte:**
- Número de funcionários: [5-15 para consultoria pequena-média]
- Faturamento anual estimado: [R$ 800k - R$ 2.5M para este porte]

**Área de Atuação Geográfica:** [Regional, Estadual, ou Multi-estadual]

**Especializações:** [3-5 áreas, baseadas nos focos das empresas semente]
Exemplos: consultoria ESG, certificações, treinamentos, mercado de carbono, etc.

**Segmentos de Clientes:** [3-4 tipos de clientes principais]

**Diferenciais Competitivos:** [2-3 pontos fortes únicos]

**Estrutura da Equipe:**
- Perfil da equipe técnica
- Principais expertises

**Parcerias Estratégicas:** [2-3 parcerias típicas: universidades, certificadoras, etc.]

**Métricas de Impacto (últimos 12 meses):**
- Clientes atendidos: [número realista para o porte]
- Projetos executados: [número realista]
- Horas de treinamento: [número realista]
- CO2e evitado através de projetos: [número realista]

REQUISITOS CRÍTICOS:
- Todos os elementos devem ser COERENTES entre si
- Porte da empresa deve ser consistente com faturamento e métricas
- Área geográfica deve ser consistente com número de clientes
- Especializações devem ser consistentes com métricas de impacto
- Use terminologia e contexto brasileiro

FORMATO: Liste cada campo claramente. Seja específico e realista.
"""
    
    return task_spec + seed_section + instructions


def generate_company_profile(model_client: OllamaClient, seed_companies: List[Dict]) -> Dict[str, Any]:
    """
    Generate a unified company profile for consistent document generation.
    
    Args:
        model_client: OllamaClient instance
        seed_companies: Seed company data
    
    Returns:
        Dictionary containing company profile data
    """
    print("GENERATING UNIFIED COMPANY PROFILE")
    
    prompt = create_company_profile_prompt(seed_companies)
    
    profile_text = model_client.generate(
        prompt=prompt,
        max_tokens=1024,
        temperature=0.7
    )
    
    print("✓ Company profile generated")
    print(profile_text)
    
    return {
        "profile_text": profile_text,
        "generation_timestamp": datetime.now().isoformat()
    }


# Generate the company profile
company_profile = generate_company_profile(model_client, seed_companies)

GENERATING UNIFIED COMPANY PROFILE
✓ Company profile generated
**Nome da Empresa:** Sustenta Estratégias Consultoria

**Localização:** Curitiba, Paraná

**Ano de Fundação:** 2018

**Porte:**
- Número de funcionários: 9
- Faturamento anual estimado: R$ 1.200.000

**Área de Atuação Geográfica:** Regional (Foco no Sul do Brasil, com expansão para o Sudeste)

**Especializações:**
1.  Consultoria ESG (Avaliação, Diagnóstico e Plano de Ação)
2.  Certificações (ISO 14001, GRI, RAEE)
3.  Treinamentos em Sustentabilidade
4.  Análise de Ciclo de Vida (ACV)

**Segmentos de Clientes:**
1.  Pequenas e médias empresas (PMEs) do setor alimentício
2.  Startups de tecnologia com foco em sustentabilidade
3.  Cooperativas de produtores rurais
4.  Organizações da sociedade civil (ONGs)

**Diferenciais Competitivos:**
1.  Abordagem personalizada e foco nas especificidades regionais.
2.  Expertise em sustentabilidade do agronegócio.
3.  Utilização de ferramentas de análise de ciclo de vida (ACV) para avalia

## 5. Document 1: Mission, Vision & Values Generator

### 5.1 MVV Prompt

In [ ]:
def create_mvv_prompt(seed_companies: List[Dict], company_profile: str) -> str:
    """
    Create seed-guided prompt for Mission, Vision & Values generation.
    
    Args:
        seed_companies: List of seed company data
        company_profile: Generated company profile
    
    Returns:
        Formatted prompt string
    """
    
    # Task specification
    task_spec = """Você é um especialista em estratégia organizacional e branding corporativo 
para empresas brasileiras de consultoria ESG/sustentabilidade. Sua tarefa é criar uma declaração 
de Missão, Visão e Valores que sejam CONSISTENTES com o perfil organizacional fornecido."""
    
    # Company profile section
    profile_section = f"""\n\n=== PERFIL DA EMPRESA (BASE OBRIGATÓRIA) ===

{company_profile}

ATENÇÃO: Sua declaração de Missão, Visão e Valores DEVE ser consistente com TODOS os 
elementos deste perfil (localização, porte, especializações, clientes, etc.)
"""
    
    # Seed companies (in-context examples)
    seed_section = "\n\n=== EMPRESAS DE REFERÊNCIA (Exemplos do Setor ESG no Brasil) ===\n"
    
    for i, company in enumerate(seed_companies, 1):
        seed_section += f"\n--- Empresa {i}: {company['company']} ---\n"
        seed_section += f"Missão: {company['mission']}\n"
        if company['vision']:
            seed_section += f"Visão: {company['vision']}\n"
        seed_section += f"Valores: {', '.join(company['values'])}\n"
    
    # Generation instructions
    instructions = """\n\n=== TAREFA ===

Com base no perfil da empresa e nos exemplos acima, gere:

**Missão:** [1-2 frases que reflitam o propósito DA EMPRESA DO PERFIL]
- Deve mencionar especializações do perfil
- Deve ser coerente com segmentos de clientes do perfil

**Visão:** [1-2 frases aspiracionais coerentes com o perfil]
- Deve refletir área de atuação geográfica do perfil
- Deve ser ambiciosa mas realista para o porte da empresa

**Valores:** [4-6 valores]
- Devem refletir os diferenciais competitivos do perfil
- Devem ser consistentes com a cultura organizacional implícita no perfil

REQUISITOS CRÍTICOS:
- COERÊNCIA TOTAL com o perfil fornecido
- NÃO copie diretamente os exemplos - gere conteúdo NOVO mas consistente
- Mantenha a distribuição de número de valores (3-5 valores, conforme exemplos)
- Use português brasileiro formal apropriado para documentos corporativos
- Preserve temas comuns observados (sustentabilidade, impacto, transparência, inovação)

FORMATO:
**Missão:**
[texto]

**Visão:**
[texto]

**Valores:**
- [valor 1]
- [valor 2]
- [valor 3]
- [valor 4]
[...]

-----
CRITICAL: Seja direto e sem preâmbulos.
"""
    
    return task_spec + profile_section + seed_section + instructions


# Test prompt creation
mvv_prompt = create_mvv_prompt(seed_companies, company_profile['profile_text'])
print("✓ Mission/Vision/Values prompt created")
print(f"  Prompt length: {len(mvv_prompt)} characters")

✓ Mission/Vision/Values prompt created
  Prompt length: 6639 characters


### 5.2 Generate MVV

In [61]:
def generate_mvv(model_client: OllamaClient, prompt: str) -> str:
    """
    Generate synthetic Mission, Vision & Values document.
    
    Args:
        model_client: OllamaClient instance
        prompt: Seed-guided generation prompt
    
    Returns:
        Generated MVV text
    """
    
    print("GENERATING DOCUMENT 1: Mission, Vision & Values")
    
    generated = model_client.generate(
        prompt=prompt,
        max_tokens=1024,  # Sufficient for MVV based on seed patterns
        temperature=0.7  # Balance between diversity and seed alignment
    )
    
    print("✓ Generation complete")
    return generated


# Generate the document
mvv_document = generate_mvv(model_client, mvv_prompt)
print(mvv_document)

GENERATING DOCUMENT 1: Mission, Vision & Values
✓ Generation complete
**Missão:** Sustenta Estratégias Consultoria impulsiona organizações do Sul do Brasil e além a alcançar a excelência em sustentabilidade, oferecendo soluções personalizadas em ESG, certificações e análise de ciclo de vida, com foco em agronegócio e startups inovadoras.

**Visão:** Ser reconhecida como líder regional em consultoria ESG, sinônimo de impacto e inovação, transformando negócios e comunidades através de práticas sustentáveis e gerando valor a longo prazo para nossos clientes e para o meio ambiente.

**Valores:**
- Impacto
- Inovação
- Transparência
- Confiança
- Resultados



## 6. DRE

## 7. Document 3: Social Impact Report Generator

**NOTE ON SEED DATA LIMITATIONS**:
The current seed companies have missions mentioning impact themes, but lack detailed impact report examples. This is acceptable for initial generation following the "augmented seed-guided" approach from the Survey paper (knowledge enhancement strategy).

### 7.1 Social Impact Report Prompt

In [66]:
def create_impact_report_prompt(seed_companies: List[Dict], 
                                company_profile: str,
                                mvv_document: str) -> str:
    """
    Create seed-guided prompt for Social Impact Report generation.
    
    NOTE: Current seed data lacks complete impact report examples.
    TODO: Find 3-5 actual Brazilian ESG impact reports (from Sistema B 
          annual reports or similar) to improve seed guidance.
        
    Args:
        seed_companies: Seed company data
    
    Returns:
        Formatted prompt string
    """
    
    # Task specification
    task_spec = """Você é um especialista em relatórios de impacto ESG. Sua tarefa é criar 
um Relatório de Impacto Social que seja TOTALMENTE CONSISTENTE com o perfil organizacional 
e a missão/visão/valores já definidos."""
    
    # Profile and MVV context
    context_section = f"""

=== PERFIL DA EMPRESA (BASE OBRIGATÓRIA) ===

{company_profile}

=== MISSÃO, VISÃO E VALORES DA EMPRESA ===

{mvv_document}

ATENÇÃO: O relatório de impacto DEVE:
- Usar as métricas exatas do perfil
- Refletir as especializações mencionadas
- Demonstrar progresso alinhado com a missão
- Citar parcerias mencionadas no perfil
"""
    
    # Seed themes
    seed_section = "\n\n=== TEMAS DE IMPACTO (Referência do Setor) ===\n"
    for i, company in enumerate(seed_companies[:3], 1):
        seed_section += f"{i}. {company['mission'][:100]}...\n"
    
    # Instructions
    instructions = """\n\n=== TAREFA ===

Gere um Relatório de Impacto Social para a empresa do perfil acima.

ESTRUTURA:

**1. MENSAGEM DA LIDERANÇA** (2 parágrafos)
- Reflita a missão e visão da empresa
- Destaque alinhamento com valores

**2. SOBRE A EMPRESA** (1 parágrafo)
- Use EXATAMENTE as informações do perfil (localização, ano, porte)

**3. INDICADORES DE IMPACTO**
USE AS MÉTRICAS EXATAS DO PERFIL:
- Clientes atendidos: [número do perfil]
- Projetos executados: [número do perfil]  
- Horas de treinamento: [número do perfil]
- CO2e evitado: [número do perfil]

**4. DESTAQUES QUALITATIVOS** (3 exemplos)
- Projetos alinhados com as especializações do perfil
- Demonstre impacto nos segmentos de clientes mencionados

**5. STAKEHOLDERS**
- Mencione as parcerias listadas no perfil

**6. COMPROMISSOS FUTUROS** (3 itens)
- Coerentes com a visão da empresa

REQUISITOS CRÍTICOS:
- CONSISTÊNCIA TOTAL com perfil e MVV
- Use números EXATOS do perfil (não invente novos)
- Mencione localização e área geográfica corretamente

FORMATO: Relatório estruturado em Markdown e com seções claras.

-----
CRITICAL: Seja direto e sem preâmbulos. Comece com:

# Relatório de Impacto Social ...
"""
    
    return task_spec + context_section + seed_section + instructions


# Create prompt
impact_report_prompt = create_impact_report_prompt(seed_companies, 
    company_profile['profile_text'],
    mvv_document
)
print("✓ Social Impact Report prompt created")
print(f"  Prompt length: {len(impact_report_prompt)} characters")

✓ Social Impact Report prompt created
  Prompt length: 4705 characters


### 7.2 Generate Social Impact Report

In [67]:
def generate_impact_report(model_client: OllamaClient, prompt: str) -> str:
    """
    Generate synthetic Social Impact Report.
    
    Args:
        model_client: OllamaClient instance
        prompt: Seed-guided generation prompt
    
    Returns:
        Generated impact report text
    """
    
    print("GENERATING DOCUMENT 3: Social Impact Report")
    
    generated = model_client.generate(
        prompt=prompt,
        max_tokens=2048,
        temperature=0.7
    )
    
    print("✓ Generation complete")
    return generated


# Generate the document
impact_report_document = generate_impact_report(model_client, impact_report_prompt)
print(impact_report_document)

GENERATING DOCUMENT 3: Social Impact Report
✓ Generation complete
# Relatório de Impacto Social - Sustenta Estratégias Consultoria

**Introdução**

A Sustenta Estratégias Consultoria tem como missão impulsionar organizações do Sul do Brasil e além a alcançar a excelência em sustentabilidade. Nossa visão é ser reconhecida como líder regional em consultoria ESG, sinônimo de impacto e inovação, transformando negócios e comunidades através de práticas sustentáveis. Estamos comprometidos com os valores de Impacto, Inovação, Transparência, Confiança e Resultados. Este relatório apresenta nosso impacto nos últimos 12 meses, refletindo nosso compromisso com a sustentabilidade e o desenvolvimento de nossos clientes e do meio ambiente.

**Sobre a Sustenta Estratégias Consultoria**

Fundada em Curitiba, Paraná, em 2018, a Sustenta Estratégias Consultoria é uma empresa de pequeno porte, com 9 funcionários e um faturamento anual estimado em R$ 1.200.000. Especializamo-nos em consultoria ESG, certif

## 8. Document 4: Business Model Canvas Generator

**NOTE ON SEED DATA LIMITATIONS**:
Seed companies only provide mission/values, not detailed business model information. Using industry standards (Osterwalder framework) + seed data for terminology.

### 8.1 Business Model Canvas Prompt

In [69]:
def create_bmc_prompt(seed_companies: List[Dict],
                        company_profile: str,
                        mvv_document: str) -> str:
    """
    Create seed-guided prompt for Business Model Canvas generation.
    
    NOTE: Seed data lacks complete business model details.
    TODO: Find 2-3 public Business Model Canvas examples from Brazilian 
          ESG/sustainability companies to improve generation quality.
    
    Uses Osterwalder framework (standard) + seed data for terminology.
    
    Args:
        seed_companies: Seed company data
    
    Returns:
        Formatted prompt string
    """
    
    # Task specification
    task_spec = """Você é um especialista em Business Model Canvas para empresas ESG brasileiras. 
Sua tarefa é criar um BMC TOTALMENTE CONSISTENTE com o perfil, missão e modelo de negócio 
da empresa."""
    
    # Context
    context_section = f"""

=== PERFIL DA EMPRESA (BASE OBRIGATÓRIA) ===

{company_profile}

=== MISSÃO E VISÃO DA EMPRESA ===

{mvv_document}

ATENÇÃO: O BMC DEVE refletir EXATAMENTE:
- Especializações → Proposta de Valor
- Segmentos de clientes mencionados → Segmentos de Clientes
- Parcerias listadas → Parcerias Principais
- Porte e faturamento → Estrutura de Custos e Fontes de Receita
- Área geográfica → Canais e Relacionamento
"""
    
    # Seed context
    seed_section = "\n\n=== EXEMPLOS DO SETOR (Referência) ===\n"
    for i, company in enumerate(seed_companies[:3], 1):
        seed_section += f"{i}. {company['company']}: {company['mission'][:80]}...\n"
    
    # Instructions
    instructions = """

=== TAREFA ===

Gere um Business Model Canvas para a empresa do perfil acima.

Para cada bloco (3-5 itens):

**1. SEGMENTOS DE CLIENTES**
- Use EXATAMENTE os segmentos mencionados no perfil

**2. PROPOSTA DE VALOR**
- Baseie-se nas especializações do perfil
- Reflita a missão da empresa

**3. CANAIS**
- Coerentes com área geográfica do perfil

**4. RELACIONAMENTO COM CLIENTES**
- Apropriado para o porte da empresa

**5. FONTES DE RECEITA**
- Alinhadas com especializações
- Realistas para o faturamento mencionado no perfil

**6. RECURSOS PRINCIPAIS**
- Inclua equipe técnica mencionada no perfil
- Inclua parcerias estratégicas do perfil

**7. ATIVIDADES-CHAVE**
- Derivadas das especializações

**8. PARCERIAS PRINCIPAIS**
- Use EXATAMENTE as parcerias listadas no perfil
- Adicione 1-2 coerentes com o setor

**9. ESTRUTURA DE CUSTOS**
- Realista para empresa do porte mencionado

REQUISITOS CRÍTICOS:
- CONSISTÊNCIA TOTAL com perfil, MVV e métricas
- Não invente informações que contradigam o perfil

FORMATO:
## BUSINESS MODEL CANVAS

### [Bloco]
- Item 1
- Item 2
[...]

-----
CRITICAL: Seja direto e sem preâmbulos. 
"""
    
    return task_spec + context_section + seed_section + instructions


# Create prompt
bmc_prompt = create_bmc_prompt(seed_companies,
    company_profile['profile_text'],
    mvv_document
)
print("✓ Business Model Canvas prompt created")
print(f"  Prompt length: {len(bmc_prompt)} characters")

✓ Business Model Canvas prompt created
  Prompt length: 4779 characters


### 8.2 Generate Business Model Canvas

In [70]:
def generate_bmc(model_client: OllamaClient, prompt: str) -> str:
    """
    Generate synthetic Business Model Canvas.
    
    Args:
        model_client: OllamaClient instance
        prompt: Seed-guided generation prompt
    
    Returns:
        Generated BMC text
    """
    print("GENERATING DOCUMENT 4: Business Model Canvas")
    
    generated = model_client.generate(
        prompt=prompt,
        max_tokens=1536,
        temperature=0.7
    )
    
    print("✓ Generation complete")
    return generated


# Generate the document
bmc_document = generate_bmc(model_client, bmc_prompt)
print(bmc_document)

GENERATING DOCUMENT 4: Business Model Canvas
✓ Generation complete
## BUSINESS MODEL CANVAS
### 1. SEGMENTOS DE CLIENTES
- Pequenas e médias empresas (PMEs) do setor alimentício
- Startups de tecnologia com foco em sustentabilidade
- Cooperativas de produtores rurais
- Organizações da sociedade civil (ONGs)

### 2. PROPOSTA DE VALOR
- Avaliação, diagnóstico e plano de ação ESG personalizados
- Serviços de certificação (ISO 14001, GRI, RAEE)
- Treinamentos em sustentabilidade
- Análise de Ciclo de Vida (ACV) para avaliação de impacto
- Soluções inovadoras para agronegócio e startups
- Abordagem regional com foco nas especificidades do Sul do Brasil

### 3. CANAIS
- Reuniões presenciais e virtuais
- Website e redes sociais (LinkedIn, Instagram)
- Eventos e feiras do setor
- Parcerias com universidades e associações
- Canais de comunicação direta (e-mail, telefone)

### 4. RELACIONAMENTO COM CLIENTES
- Consultoria personalizada e contínua
- Suporte técnico e acompanhamento de projetos
- C

## 9. SWOT Analysis Generator

### 9.1 SWOT Prompt

In [74]:
def create_swot_prompt(seed_companies: List[Dict],
                        company_profile: str,
                        mvv_document: str,
                        bmc_document: str) -> str:
    """
    Create seed-guided prompt for SWOT Analysis generation.
    
    Args:
        seed_companies: List of real company examples
        company_profile: Generated company profile
        mvv_document: Generated MVV
        bmc_document: Generated BMC
    
    Returns:
        Formatted prompt string    
    """
    
    # Task specification
    task_spec = """Você é um especialista em análise estratégica para empresas ESG brasileiras. 
Sua tarefa é criar uma análise SWOT TOTALMENTE CONSISTENTE com o perfil, missão, visão, valores 
e modelo de negócio já definidos."""
    
    # Full context
    context_section = f"""

=== PERFIL DA EMPRESA ===

{company_profile}

=== MISSÃO, VISÃO E VALORES ===

{mvv_document}

=== MODELO DE NEGÓCIO (BMC) ===

{bmc_document[:800]}... [resumo]

ATENÇÃO: O SWOT DEVE:
- FORÇAS: Refletir diferenciais e recursos do perfil/BMC
- FRAQUEZAS: Considerar limitações do porte e área geográfica
- OPORTUNIDADES: Alinhadas com visão e especializações
- AMEAÇAS: Relevantes para o mercado e posicionamento da empresa
"""
    
    # Market context
    market_section = """\n\n=== CONTEXTO DE MERCADO ESG BRASILEIRO ===

- Competição crescente
- Demanda regulatória aumentando
- Entrada de grandes consultorias
- Oportunidades em carbono e certificações
"""
    
    # Instructions
    instructions = """\n\n=== TAREFA ===

Gere uma análise SWOT consistente com TODOS os documentos anteriores.

**FORÇAS** (5 itens)
- Derive dos "Diferenciais Competitivos" e "Recursos Principais" do BMC
- Mencione parcerias estratégicas do perfil

**FRAQUEZAS** (5 itens)
- Considere o porte da empresa
- Considere limitações geográficas mencionadas

**OPORTUNIDADES** (5 itens)
- Alinhadas com a VISÃO da empresa
- Relacionadas às especializações

**AMEAÇAS** (5 itens)
- Relevantes para o setor e posicionamento

REQUISITOS CRÍTICOS:
- CONSISTÊNCIA com perfil, MVV e BMC
- Não contradiga informações anteriores
- Seja específico e realista

FORMATO:
## ANÁLISE SWOT

### FORÇAS
- [item]
[...]

### FRAQUEZAS
- [item]
[...]

### OPORTUNIDADES
- [item]
[...]

### AMEAÇAS
- [item]
[...]

-----
CRITICAL: Seja direto e sem preâmbulos. 
"""
    
    return task_spec + context_section + market_section + instructions


# Create SWOT prompt
swot_prompt = create_swot_prompt(seed_companies,
    company_profile['profile_text'],
    mvv_document,
    bmc_document
)
print("✓ SWOT Analysis prompt created")
print(f"  Prompt length: {len(swot_prompt)} characters")

✓ SWOT Analysis prompt created
  Prompt length: 5078 characters


### 9.2 Generate SWOT Analysis

In [75]:
def generate_swot(model_client: OllamaClient, prompt: str) -> str:
    """
    Generate synthetic SWOT Analysis document.
    
    Args:
        model_client: OllamaClient instance
        prompt: Seed-guided generation prompt
    
    Returns:
        Generated SWOT analysis text
    """
    print("GENERATING DOCUMENT 5: SWOT Analysis")
    
    generated = model_client.generate(
        prompt=prompt,
        max_tokens=1536,
        temperature=0.7
    )
    
    print("✓ Generation complete")
    return generated


# Generate the document
swot_document = generate_swot(model_client, swot_prompt)
print(swot_document)

GENERATING DOCUMENT 5: SWOT Analysis
✓ Generation complete
## ANÁLISE SWOT - Sustenta Estratégias Consultoria

### FORÇAS

1.  **Expertise Regionalizada:** Profundo conhecimento do agronegócio do Sul do Brasil, crucial para atender às necessidades específicas da região.
2.  **Análise de Ciclo de Vida (ACV):** Diferenciação através da aplicação de ACV, fornecendo avaliações de impacto detalhadas e valiosas.
3.  **Abordagem Personalizada:** Serviços de consultoria ESG sob medida para cada cliente, aumentando a relevância e eficácia.
4.  **Parcerias Estratégicas:** Colaborações com UFPR, IBS e CertificaBrasil impulsionam inovação e acesso a recursos.
5.  **Equipe Especializada:** Ana Paula, Pedro e Sofia trazem competências complementares em sustentabilidade, certificações e análise de dados.

### FRAQUEZAS

1.  **Porte da Empresa:** Número limitado de funcionários e faturamento podem restringir recursos e alcance em comparação com grandes consultorias.
2.  **Foco Geográfico:** Principalm

## 6. Save Generated Documents

In [76]:
def save_documents(mvv: str, swot: str, impact: str, bmc: str, 
                       output_dir: str = "synthetic_documents"):
    """
    Save all generated synthetic documents.
    
    Args:
        mvv: Mission/Vision/Values text
        swot: SWOT Analysis text
        impact: Social Impact Report text
        bmc: Business Model Canvas text
        output_dir: Output directory path
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Save Document 1: MVV
    with open(os.path.join(output_dir, "doc1_mission_vision_values.md"), 'w', encoding='utf-8') as f:
        f.write(f"# Missão, Visão e Valores\n\n")
        f.write(f"*Documento Sintético | Gerado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
        f.write(mvv)
    
    # Save Document 3: Impact Report
    with open(os.path.join(output_dir, "doc3_social_impact_report.md"), 'w', encoding='utf-8') as f:
        f.write(f"# Relatório de Impacto Social\n\n")
        f.write(f"*Documento Sintético | Gerado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
        f.write(impact)
    
    # Save Document 4: BMC
    with open(os.path.join(output_dir, "doc4_business_model_canvas.md"), 'w', encoding='utf-8') as f:
        f.write(f"# Business Model Canvas\n\n")
        f.write(f"*Documento Sintético | Gerado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
        f.write(bmc)
    
    # Save Document 5: SWOT
    with open(os.path.join(output_dir, "doc5_swot_analysis.md"), 'w', encoding='utf-8') as f:
        f.write(f"# Análise SWOT\n\n")
        f.write(f"*Documento Sintético | Gerado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
        f.write(swot)
    
    # Update metadata
    metadata = {
        "generation_date": datetime.now().isoformat(),
        "model": "brunoconterato/Gemma-3-Gaia-PT-BR-4b-it:f16",
        "methodology": "Seed-guided synthetic data generation",
        "seed_companies_count": len(seed_companies),
        "seed_companies_sector": "Brazilian ESG Consulting",
        "documents_generated": [
            "Mission_Vision_Values",
            "Social_Impact_Report",
            "Business_Model_Canvas",
            "SWOT_Analysis"
        ],
        "generation_parameters": {
            "temperature": 0.7,
            "random_seed": 42
        },
        "notes": {
            "doc1": "Pure seed-guided generation",
            "doc3": "Augmented approach - seed themes + GRI framework structure",
            "doc4": "Framework-guided - Osterwalder BMC + seed terminology",
            "doc5": "Pure seed-guided generation"
        },
        "references": [
            "DocGenie: A Framework for High-Fidelity Synthetic Document Generation (Harikrishnan et al., 2025)",
            "On LLMs-Driven Synthetic Data Generation, Curation, and Evaluation: A Survey (Long et al., 2024)",
            "SynEval: A Multi-Faceted Evaluation Framework (Paper 3)"
        ]
    }
    
    metadata_path = os.path.join(output_dir, "generation_metadata.json")
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print(f"\n✓ Synthetic documents saved to: {output_dir}/")


# Save all documents
save_documents(mvv_document, swot_document, impact_report_document, bmc_document)


✓ Synthetic documents saved to: synthetic_documents/


## 10. Validation

### 10.1 Cross-Document Consistency Validation

In [77]:
def validate_cross_document_consistency(
    company_profile: str,
    mvv: str,
    impact: str,
    bmc: str,
    swot: str
) -> Dict[str, Any]:
    """
    Validate cross-document consistency.
    
    Args:
        company_profile: Generated company profile
        mvv: Mission/Vision/Values
        impact: Social Impact Report
        bmc: Business Model Canvas
        swot: SWOT Analysis
    
    Returns:
        Dictionary with consistency check results
    """
    
    print("\n" + "="*70)
    print("VALIDATION: Cross-Document Consistency")
    print("="*70)
    
    results = {
        "checks_passed": 0,
        "checks_total": 0,
        "details": {}
    }
    
    # Extract company name from profile (if present)
    profile_lower = company_profile.lower()
    
    # Check 1: Company size consistency
    print("\n📋 Check 1: Company Size Consistency")
    size_mentions = []
    if "pequena" in profile_lower or "pequeno" in profile_lower:
        size_mentions.append("profile: pequena")
    if "média" in profile_lower or "medio" in profile_lower:
        size_mentions.append("profile: média")
    
    # Check if SWOT mentions size appropriately
    swot_lower = swot.lower()
    swot_mentions_size = ("pequena" in swot_lower or "média" in swot_lower or 
                          "porte" in swot_lower or "recurso" in swot_lower)
    
    size_consistent = len(size_mentions) > 0
    results["checks_total"] += 1
    if size_consistent:
        results["checks_passed"] += 1
        print(f"  ✓ PASS: Size indicators found - {size_mentions}")
    else:
        print(f"  ⚠ WARNING: Size not clearly specified")
    results["details"]["size_consistency"] = size_consistent
    
    # Check 2: Geographic scope consistency
    print("\n📋 Check 2: Geographic Scope Consistency")
    geo_keywords = ["regional", "estadual", "nacional", "rio grande do sul", 
                    "sul", "brasil", "estado"]
    
    geo_in_profile = sum(1 for kw in geo_keywords if kw in profile_lower)
    geo_in_mvv = sum(1 for kw in geo_keywords if kw in mvv.lower())
    geo_in_bmc = sum(1 for kw in geo_keywords if kw in bmc.lower())
    
    geo_consistent = geo_in_profile > 0 and (geo_in_mvv > 0 or geo_in_bmc > 0)
    results["checks_total"] += 1
    if geo_consistent:
        results["checks_passed"] += 1
        print(f"  ✓ PASS: Geographic scope mentioned across documents")
    else:
        print(f"  ⚠ WARNING: Geographic scope may not be consistent")
    results["details"]["geography_consistency"] = geo_consistent
    
    # Check 3: Service/Specialization alignment
    print("\n📋 Check 3: Service/Specialization Alignment")
    service_keywords = ["consultoria", "treinamento", "capacitação", "carbono",
                       "certificação", "auditoria", "esg", "sustentabilidade"]
    
    services_in_profile = [kw for kw in service_keywords if kw in profile_lower]
    services_in_mvv = [kw for kw in service_keywords if kw in mvv.lower()]
    services_in_bmc = [kw for kw in service_keywords if kw in bmc.lower()]
    services_in_impact = [kw for kw in service_keywords if kw in impact.lower()]
    
    # Check overlap
    common_services = set(services_in_profile) & (
        set(services_in_mvv) | set(services_in_bmc) | set(services_in_impact)
    )
    
    service_consistent = len(common_services) >= 2
    results["checks_total"] += 1
    if service_consistent:
        results["checks_passed"] += 1
        print(f"  ✓ PASS: Common services found - {common_services}")
    else:
        print(f"  ⚠ WARNING: Service alignment may be weak")
    results["details"]["service_consistency"] = service_consistent
    
    # Check 4: Values reflection in other documents
    print("\n📋 Check 4: Values Reflected in Other Documents")
    
    # Extract values from MVV
    values_section = mvv.lower().split("valores:")[-1] if "valores:" in mvv.lower() else ""
    value_keywords = ["transparência", "inovação", "ética", "sustentabilidade",
                     "compromisso", "excelência", "confiança", "impacto"]
    
    values_in_mvv = [kw for kw in value_keywords if kw in values_section]
    values_in_swot = [kw for kw in value_keywords if kw in swot.lower()]
    values_in_impact = [kw for kw in value_keywords if kw in impact.lower()]
    
    values_reflected = len(set(values_in_mvv) & (set(values_in_swot) | set(values_in_impact))) >= 1
    results["checks_total"] += 1
    if values_reflected:
        results["checks_passed"] += 1
        print(f"  ✓ PASS: Values reflected in other documents")
    else:
        print(f"  ⚠ WARNING: Values may not be well-integrated")
    results["details"]["values_reflection"] = values_reflected
    
    # Overall consistency score
    consistency_score = results["checks_passed"] / results["checks_total"] if results["checks_total"] > 0 else 0
    
    print(f"\n📊 OVERALL CONSISTENCY SCORE: {consistency_score*100:.1f}%")
    print(f"    Checks passed: {results['checks_passed']}/{results['checks_total']}")
    print(f"    Target: ≥75% for acceptable coherence")
    
    if consistency_score >= 0.75:
        print("    ✓ PASS: Documents show good cross-consistency")
    else:
        print("    ⚠ WARNING: Some consistency issues detected")
    
    results["consistency_score"] = consistency_score
    print("="*70)
    
    return results


# Run cross-document validation
consistency_results = validate_cross_document_consistency(
    company_profile['profile_text'],
    mvv_document,
    impact_report_document,
    bmc_document,
    swot_document
)


VALIDATION: Cross-Document Consistency

📋 Check 1: Company Size Consistency
  ✓ PASS: Size indicators found - ['profile: pequena', 'profile: média']

📋 Check 2: Geographic Scope Consistency
  ✓ PASS: Geographic scope mentioned across documents

📋 Check 3: Service/Specialization Alignment
  ✓ PASS: Common services found - {'capacitação', 'esg', 'certificação', 'treinamento', 'sustentabilidade', 'consultoria'}

📋 Check 4: Values Reflected in Other Documents
  ✓ PASS: Values reflected in other documents

📊 OVERALL CONSISTENCY SCORE: 100.0%
    Checks passed: 4/4
    Target: ≥75% for acceptable coherence
    ✓ PASS: Documents show good cross-consistency


### 10.2 Structure Preservation Score
Implementing fidelity metrics from SynEval paper (Paper 3)

In [78]:
def validate_structure_preservation(mvv: str, impact: str, bmc: str, swot: str):
    """
    Compute Structure Preservation Score (SPS) from SynEval framework.
    
    Args:
        mvv: Mission/Vision/Values text
        impact: Social Impact Report text
        bmc: Business Model Canvas text
        swot: SWOT Analysis text
    """
    
    print("\n" + "="*70)
    print("VALIDATION: Structure Preservation Score (SPS)")
    print("="*70)
    
    results = {}
    
    # Document 1: MVV
    print("\n📄 Document 1: Mission, Vision & Values")
    mvv_elements = ["missão", "visão", "valores"]
    mvv_present = [e for e in mvv_elements if e in mvv.lower()]
    mvv_sps = len(mvv_present) / len(mvv_elements)
    results['doc1_mvv'] = mvv_sps
    print(f"  Structure Preservation Score: {mvv_sps*100:.1f}%")
    print(f"  Present elements: {mvv_present}")
    
    # Document 3: Impact Report
    print("\n📄 Document 3: Social Impact Report")
    impact_elements = ["impacto", "indicadores", "stakeholder"]
    impact_present = [e for e in impact_elements if e in impact.lower()]
    impact_sps = len(impact_present) / len(impact_elements)
    results['doc3_impact'] = impact_sps
    print(f"  Structure Preservation Score: {impact_sps*100:.1f}%")
    print(f"  Present elements: {impact_present}")
    
    # Document 4: BMC
    print("\n📄 Document 4: Business Model Canvas")
    bmc_elements = ["clientes", "valor", "receita", "recursos", "atividades", 
                    "parcerias", "custos"]
    bmc_present = [e for e in bmc_elements if e in bmc.lower()]
    bmc_sps = len(bmc_present) / len(bmc_elements)
    results['doc4_bmc'] = bmc_sps
    print(f"  Structure Preservation Score: {bmc_sps*100:.1f}%")
    print(f"  Present elements: {bmc_present} ({len(bmc_present)}/{len(bmc_elements)})")
    
    # Document 5: SWOT
    print("\n📄 Document 5: SWOT Analysis")
    swot_elements = ["forças", "fraquezas", "oportunidades", "ameaças"]
    swot_present = [e for e in swot_elements if e in swot.lower()]
    swot_sps = len(swot_present) / len(swot_elements)
    results['doc5_swot'] = swot_sps
    print(f"  Structure Preservation Score: {swot_sps*100:.1f}%")
    print(f"  Present elements: {swot_present}")
    
    # Overall
    overall_sps = sum(results.values()) / len(results)
    print(f"\n📊 Overall Structure Preservation Score: {overall_sps*100:.1f}%")
    print(f"    Target: >90% for high fidelity")
    print("="*70)
    
    return results


# Run validation
validation_results = validate_structure_preservation(
    mvv_document, 
    impact_report_document, 
    bmc_document, 
    swot_document
)


VALIDATION: Structure Preservation Score (SPS)

📄 Document 1: Mission, Vision & Values
  Structure Preservation Score: 100.0%
  Present elements: ['missão', 'visão', 'valores']

📄 Document 3: Social Impact Report
  Structure Preservation Score: 66.7%
  Present elements: ['impacto', 'indicadores']

📄 Document 4: Business Model Canvas
  Structure Preservation Score: 100.0%
  Present elements: ['clientes', 'valor', 'receita', 'recursos', 'atividades', 'parcerias', 'custos'] (7/7)

📄 Document 5: SWOT Analysis
  Structure Preservation Score: 100.0%
  Present elements: ['forças', 'fraquezas', 'oportunidades', 'ameaças']

📊 Overall Structure Preservation Score: 91.7%
    Target: >90% for high fidelity


### 10.3 Semantic Similarity Validation

Academic Basis: Sentence-BERT embeddings for fidelity assessment (Reimers & Gurevych, 2019)
This metric validates that synthetic documents are:

Domain-aligned (similar to seed companies)
Not memorized (not too similar - avoiding plagiarism)

In [82]:
def compute_semantic_similarity(
    synthetic_texts: Dict[str, str],
    seed_companies: List[Dict],
    model_name: str = 'paraphrase-multilingual-mpnet-base-v2'
) -> Dict[str, Any]:
    """
    Compute semantic similarity between synthetic documents and seed data.
    
    Validates:
    1. Fidelity: Are synthetic docs domain-aligned with seeds?
    2. Memorization: Are synthetic docs too similar (plagiarism risk)?
    
    Academic basis: Sentence-BERT (Reimers & Gurevych, 2019)
    
    Args:
        synthetic_texts: Dict of {doc_type: text} for synthetic documents
        seed_companies: Seed company data
        model_name: Sentence transformer model
    
    Returns:
        Dictionary with similarity metrics
    """
    
    print("\n" + "="*70)
    print("VALIDATION: Semantic Similarity to Seeds")
    print("="*70)
    print(f"Loading model: {model_name}")
    
    # Load sentence transformer model
    model = SentenceTransformer(model_name)
    print("✓ Model loaded")
    
    results = {}
    
    # ========================================
    # 1. MISSION SIMILARITY
    # ========================================
    print("\n📄 Document 1: Mission Similarity")
    
    # Extract missions from seeds
    seed_missions = [company['mission'] for company in seed_companies]
    
    # Extract synthetic mission
    mvv_text = synthetic_texts.get('mvv', '')
    synthetic_mission = ""
    if "missão:" in mvv_text.lower():
        mission_section = mvv_text.lower().split("missão:")[1]
        if "visão:" in mission_section:
            synthetic_mission = mission_section.split("visão:")[0].strip()
        else:
            synthetic_mission = mission_section[:200].strip()
    
    if synthetic_mission:
        # Encode
        synthetic_emb = model.encode([synthetic_mission])
        seed_embs = model.encode(seed_missions)
        
        # Compute similarities
        similarities = cosine_similarity(synthetic_emb, seed_embs)[0]
        
        mean_sim = np.mean(similarities)
        max_sim = np.max(similarities)
        std_sim = np.std(similarities)
        
        memorization_risk = (max_sim > 0.90)
        
        print(f"  Mean similarity: {mean_sim:.3f}")
        print(f"  Max similarity: {max_sim:.3f}")
        print(f"  Std deviation: {std_sim:.3f}")
        print(f"  Memorization risk (>0.90): {'YES ⚠️' if memorization_risk else 'NO ✓'}")
        
        results['mission'] = {
            'mean_similarity': float(mean_sim),
            'max_similarity': float(max_sim),
            'std_similarity': float(std_sim),
            'memorization_risk': memorization_risk
        }
    else:
        print("  ⚠️ WARNING: Could not extract mission text")
        results['mission'] = None
    
    # ========================================
    # 2. VALUES SIMILARITY
    # ========================================
    print("\n📄 Document 1: Values Similarity")
    
    # Extract values from seeds
    seed_values_texts = [", ".join(company['values']) for company in seed_companies]
    
    # Extract synthetic values
    synthetic_values_text = ""
    if "valores:" in mvv_text.lower():
        values_section = mvv_text.lower().split("valores:")[1]
        synthetic_values_text = values_section[:300].strip()
    
    if synthetic_values_text:
        # Encode
        synthetic_emb = model.encode([synthetic_values_text])
        seed_embs = model.encode(seed_values_texts)
        
        # Compute similarities
        similarities = cosine_similarity(synthetic_emb, seed_embs)[0]
        
        mean_sim = np.mean(similarities)
        max_sim = np.max(similarities)
        std_sim = np.std(similarities)
        
        memorization_risk = (max_sim > 0.90)
        
        print(f"  Mean similarity: {mean_sim:.3f}")
        print(f"  Max similarity: {max_sim:.3f}")
        print(f"  Std deviation: {std_sim:.3f}")
        print(f"  Memorization risk (>0.90): {'YES ⚠️' if memorization_risk else 'NO ✓'}")
        
        results['values'] = {
            'mean_similarity': float(mean_sim),
            'max_similarity': float(max_sim),
            'std_similarity': float(std_sim),
            'memorization_risk': memorization_risk
        }
    else:
        print("  ⚠️ WARNING: Could not extract values text")
        results['values'] = None
    
    # ========================================
    # 3. OVERALL DOCUMENT-LEVEL SIMILARITY
    # ========================================
    print("\n📄 Overall Document Similarity")
    
    # Use full MVV text
    full_synthetic_text = synthetic_texts.get('mvv', '')
    
    # Use full seed company descriptions (mission + vision + values)
    seed_full_texts = []
    for company in seed_companies:
        text = company['mission']
        if company.get('vision'):
            text += " " + company['vision']
        text += " " + ", ".join(company['values'])
        seed_full_texts.append(text)
    
    if full_synthetic_text:
        # Encode
        synthetic_emb = model.encode([full_synthetic_text])
        seed_embs = model.encode(seed_full_texts)
        
        # Compute similarities
        similarities = cosine_similarity(synthetic_emb, seed_embs)[0]
        
        mean_sim = np.mean(similarities)
        max_sim = np.max(similarities)
        std_sim = np.std(similarities)
        
        memorization_risk = (max_sim > 0.90)
        
        print(f"  Mean similarity: {mean_sim:.3f}")
        print(f"  Max similarity: {max_sim:.3f}")
        print(f"  Std deviation: {std_sim:.3f}")
        print(f"  Memorization risk (>0.90): {'YES ⚠️' if memorization_risk else 'NO ✓'}")
        
        results['overall'] = {
            'mean_similarity': float(mean_sim),
            'max_similarity': float(max_sim),
            'std_similarity': float(std_sim),
            'memorization_risk': memorization_risk
        }
    
    # ========================================
    # INTERPRETATION
    # ========================================
    print("\n" + "="*70)
    print("INTERPRETATION:")
    print("="*70)
    
    if results.get('overall'):
        mean_overall = results['overall']['mean_similarity']
        
        if 0.40 <= mean_overall <= 0.70:
            print("✓ EXCELLENT: Domain-aligned but diverse (0.40-0.70 range)")
            interpretation = "excellent"
        elif mean_overall < 0.40:
            print("⚠️ WARNING: Low similarity - may be off-topic (<0.40)")
            interpretation = "low_fidelity"
        elif mean_overall > 0.85:
            print("⚠️ WARNING: High similarity - possible memorization (>0.85)")
            interpretation = "high_memorization"
        else:
            print("✓ GOOD: Acceptable domain alignment (0.70-0.85 range)")
            interpretation = "good"
        
        results['interpretation'] = interpretation
    
    print("="*70)
    
    return results


# Run semantic similarity validation
semantic_similarity_results = compute_semantic_similarity(
    synthetic_texts={
        'mvv': mvv_document,
        'impact': impact_report_document,
        'bmc': bmc_document,
        'swot': swot_document
    },
    seed_companies=seed_companies
)


VALIDATION: Semantic Similarity to Seeds
Loading model: paraphrase-multilingual-mpnet-base-v2
✓ Model loaded

📄 Document 1: Mission Similarity
  Mean similarity: 0.510
  Max similarity: 0.801
  Std deviation: 0.165
  Memorization risk (>0.90): NO ✓

📄 Document 1: Values Similarity
  Mean similarity: 0.564
  Max similarity: 0.683
  Std deviation: 0.104
  Memorization risk (>0.90): NO ✓

📄 Overall Document Similarity
  Mean similarity: 0.589
  Max similarity: 0.750
  Std deviation: 0.093
  Memorization risk (>0.90): NO ✓

INTERPRETATION:
✓ EXCELLENT: Domain-aligned but diverse (0.40-0.70 range)


### 10.4 Human Eval